# Download Videos From Dryad

Use this notebook to fetch raw videos into `video/` before running per-date analysis notebooks.

Workflow:
1. Fill in the Dryad DOI and file manifest in Cell 2.
2. Run the download cell to populate `video/`.
3. Run checksum verification to confirm data integrity.

In [ ]:
from pathlib import Path

ROOT = Path('..').resolve()
VIDEO_DIR = ROOT / 'video'
VIDEO_DIR.mkdir(parents=True, exist_ok=True)

# Fill this in when known
DRYAD_DOI = '10.5061/dryad.x0k6djj17'

# One dict per file. Use direct download URLs from Dryad (or mirrors).
# expected_sha256 can be None temporarily, but should be set for final verification.
FILE_MANIFEST = [
    {
        'filename': 'GX010063_2025Nov10.MP4',
        'url': 'https://datadryad.org/stash/downloads/file_stream/YOUR_FILE_ID_1',
        'expected_sha256': None,
    },
    {
        'filename': 'GX010067_2025Nov12.MP4',
        'url': 'https://datadryad.org/stash/downloads/file_stream/YOUR_FILE_ID_2',
        'expected_sha256': None,
    },
    {
        'filename': 'GX010072_2025Nov13.MP4',
        'url': 'https://datadryad.org/stash/downloads/file_stream/YOUR_FILE_ID_3',
        'expected_sha256': None,
    },
    {
        'filename': 'GX01007X_2025Nov14.MP4',
        'url': 'https://datadryad.org/stash/downloads/file_stream/YOUR_FILE_ID_4',
        'expected_sha256': None,
    },
]

print('Dryad DOI:', DRYAD_DOI)
print('Target directory:', VIDEO_DIR)
print('Files listed:', len(FILE_MANIFEST))

In [ ]:
import urllib.request

def download_file(url: str, dest: Path, chunk_size: int = 1024 * 1024) -> None:
    req = urllib.request.Request(url, headers={'User-Agent': 'signal-respirometry-downloader/1.0'})
    with urllib.request.urlopen(req) as response, open(dest, 'wb') as out_f:
        while True:
            chunk = response.read(chunk_size)
            if not chunk:
                break
            out_f.write(chunk)

for item in FILE_MANIFEST:
    filename = item['filename']
    url = item['url']
    out_path = VIDEO_DIR / filename

    if out_path.exists() and out_path.stat().st_size > 0:
        print(f'Skipping existing file: {out_path.name}')
        continue

    print(f'Downloading {filename} ...')
    download_file(url, out_path)
    print(f'Wrote: {out_path} ({out_path.stat().st_size} bytes)')

In [ ]:
import hashlib

def sha256sum(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()

all_ok = True
for item in FILE_MANIFEST:
    path = VIDEO_DIR / item['filename']
    expected = item.get('expected_sha256')

    if not path.exists():
        all_ok = False
        print(f'MISSING: {path.name}')
        continue

    observed = sha256sum(path)
    if expected:
        ok = observed.lower() == expected.lower()
        all_ok = all_ok and ok
        status = 'OK' if ok else 'MISMATCH'
        print(f'{status}: {path.name}')
        if not ok:
            print('  expected:', expected)
            print('  observed:', observed)
    else:
        print(f'NO EXPECTED HASH: {path.name} -> {observed}')

if all_ok:
    print('Checksum verification completed: no mismatches detected among files with expected hashes.')
else:
    print('Checksum verification found missing files and/or mismatches.')